### Importing Libraries And Packages

In [5]:
# Importing libraries and Packages
import pandas as pd 
import numpy as np 
import requests
import io
import re

# import tkinter as tk
# import qgrid

pd.set_option('display.max_columns', None)

<h1><center> Defining Path for Importing And Exporting CSV </center></h1>

In [6]:
month_folder_name = 'May 2026'
date = '11052026'

folder_path_read = '/Users/shubham.bansla/Desktop/RelevancyLogic/'+month_folder_name+'/'+date+'/RawData_'+date+'.csv'
folder_path_out_v0_check = month_folder_name + '/'+date+'/'+'ProdRanking_'+date+'_V0.csv'

folder_path_out_v1 = month_folder_name + '/'+date+'/'+'ProdRanking_'+date+'.csv'

folder_path_out_V2 = month_folder_name + '/'+date+'/'+'ProdRanking_'+date+'_V3.csv'

folder_path_out_prepared = month_folder_name + '/'+date+'/'+'FinalRanking'+date+'.csv'

folder_path_out_seo = month_folder_name + '/'+date+'/'+'SEO'+'.csv'

folder_path_out_nudge = month_folder_name + '/'+date+'/'+'Nudge_'+date+'.csv'


print("Raw Data File From Relevancy Dashboard :- ",folder_path_read)
print()
print("Calculation File :- ",folder_path_out_v1)
print()
print("Rank File Before Clustering And Removal of L1 :- ",folder_path_out_V2)
print()
print("Rank File After Clustering And Removal of L1 :- ",folder_path_out_prepared)
print()
print("SEO File relevancy logic :- ",folder_path_out_seo)

# folder_path_out_v0  = 'March 2025/'+'31st March 2025/ProdRanking_31st_march_2025_V0.csv'



Raw Data File From Relevancy Dashboard :-  /Users/shubham.bansla/Desktop/RelevancyLogic/May 2026/11052026/RawData_11052026.csv

Calculation File :-  May 2026/11052026/ProdRanking_11052026.csv

Rank File Before Clustering And Removal of L1 :-  May 2026/11052026/ProdRanking_11052026_V3.csv

Rank File After Clustering And Removal of L1 :-  May 2026/11052026/FinalRanking11052026.csv

SEO File relevancy logic :-  May 2026/11052026/SEO.csv


In [7]:
## Reading Raw Data File 
df_1 = pd.read_csv(folder_path_read, low_memory=False)
df_1 = df_1.fillna(0)
df = df_1

<h1><center> Social Nudge Tag </center></h1>
   

In [8]:
# df = df.sort_values(by='last_15_days_sale', ascending=False).reset_index(drop=True)
# df['Social Nudge'] = ''
# df.loc[:999, 'social_nudge_tag'] = 'top 1000 quantity sold in last 30 days'

In [9]:
def sale_message(sale):
    if sale >= 1000:
        return "1000+ units sold in the last month"
    elif 500 < sale < 1000:
        return "500+ units sold in the last month"
    else:
        return ""


In [10]:
df['Social Nudge'] = df['last_30_days_sale'].apply(sale_message)

<h1><center> Defining Weights For Various Parameters </center></h1>
   

In [11]:
## AGING
aging_weight_overall = 0.66
#1.10
#1.50
## Invenyory
inventory_weight = 1

### REVENUE
# 1. Last 15 Days Revenue Weight
last_15_days_revenue_weighted_1 = 0.75
# last_15_days_revenue_weighted_1 = 0.85


## --- ******************* 

last_15_days_revenue_weighted_2 = 1.5

# 2. Last 60 Days Revenue Weight (Before last 15 days and after 75 days)
after_75_before_last_15_days_revenue_weighted = 0
#0
#### View 

last_15_days_view_weighted = 0
after_75_before_last_15_days_users_view_weighted = 0

#### Sales 
last_15_days_sale_weighted = 0.25
# last_15_days_sale_weighted = 0.15
# after_75_before_last_15_days_sale_weighted = 0.50
after_75_before_last_15_days_sale_weighted = 0.25

### RPV 
# RPV_weighted = 0.5
RPV_weighted = 0.60

### SPV 
# SPV_weighted = 0.5
SPV_weighted = 0.40

# ------------------ 

print("inventory_weights ",  aging_weight_overall+ inventory_weight + last_15_days_revenue_weighted_1) 

print("Other Variable ",  
      last_15_days_revenue_weighted_2+ after_75_before_last_15_days_revenue_weighted + last_15_days_view_weighted+after_75_before_last_15_days_users_view_weighted+last_15_days_sale_weighted + after_75_before_last_15_days_sale_weighted + RPV_weighted + SPV_weighted ) 



inventory_weights  2.41
Other Variable  3.0


<h1><center> Feature Engineering </center></h1>

### Variable 1 - Product Aging


In [12]:
# Product Aging Fraction 
df['product_aging_value'] = df['product_aging']/ df['avg_product_aging']
df['product_aging_value'].replace([np.inf, -np.inf], 0, inplace=True)

# Max - Scaling
df['product_aging_value'] = 1 - (df['product_aging_value'] / df['product_aging_value'].max())

df['product_aging_value'] = np.where(df['product_aging'] > 20, df['product_aging_value'] == 0 , df['product_aging_value'])

df['product_aging_weighted'] = df['product_aging_value'] * aging_weight_overall

# df['product_aging_weighted'] = np.where(df['product_aging'] > 45, df['product_aging_weighted'] == 1 , df['product_aging_weighted'])




/var/folders/7w/pnb04fl57d30_b6_35c_x4xwzktnc0/T/ipykernel_7346/2657492259.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['product_aging_value'].replace([np.inf, -np.inf], 0, inplace=True)


### Variable 2 - Inventory 

In [13]:
df['inventory_value'] = df['cbrt_inventory_left']/ df['cbrt_inventory_left'].max()

df['inventory_weighted'] = df['inventory_value'] * inventory_weight

### Variable 3- Last 15 Days Revenue

In [14]:

df ['last_15_days_revenue_value'] = df['cbrt_last_15_days_revenue']/df['avg_last_15_days_revenue']
df['last_15_days_revenue_value'].replace([np.inf, -np.inf], 0, inplace=True)
df['last_15_days_revenue_value'] = df['last_15_days_revenue_value'] / df['last_15_days_revenue_value'].max()

# For Inventory Sales Rank
df['last_15_days_revenue_weighted_1'] = df['last_15_days_revenue_value'] * last_15_days_revenue_weighted_1
# For Other Variables Rank
df['last_15_days_revenue_weighted_2'] = df['last_15_days_revenue_value'] * last_15_days_revenue_weighted_2


/var/folders/7w/pnb04fl57d30_b6_35c_x4xwzktnc0/T/ipykernel_7346/3856773286.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['last_15_days_revenue_value'].replace([np.inf, -np.inf], 0, inplace=True)


### Variable 4- After 75 Before Last 15 Days Revenue

In [15]:
df['after_75_before_last_15_days_revenue_value'] = df['cbrt_after_75_before_last_15_days_revenue'] / df['avg_after_75_before_last_15_days_revenue']
df['after_75_before_last_15_days_revenue_value'].replace([np.inf, -np.inf], 0, inplace=True)
df['after_75_before_last_15_days_revenue_value'] = df['after_75_before_last_15_days_revenue_value']/ df['after_75_before_last_15_days_revenue_value'].max()
df['after_75_before_last_15_days_revenue_weighted'] = df['after_75_before_last_15_days_revenue_value'] * after_75_before_last_15_days_revenue_weighted


/var/folders/7w/pnb04fl57d30_b6_35c_x4xwzktnc0/T/ipykernel_7346/4094917496.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['after_75_before_last_15_days_revenue_value'].replace([np.inf, -np.inf], 0, inplace=True)


### Variable 5 - Last 15 Days User View

In [16]:
df['last_15_days_users_view_value'] = df['cbrt_last_15_days_users_view']/df['avg_last_15_days_users_view']
df['last_15_days_users_view_value'].replace([np.inf, -np.inf], 0, inplace=True)
df['last_15_days_users_view_value'] = df['last_15_days_users_view_value']/ df['last_15_days_users_view_value'].max()
df['last_15_days_users_view_weighted'] = df['last_15_days_users_view_value'] * last_15_days_view_weighted

/var/folders/7w/pnb04fl57d30_b6_35c_x4xwzktnc0/T/ipykernel_7346/3541096349.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['last_15_days_users_view_value'].replace([np.inf, -np.inf], 0, inplace=True)


### Variable 6 - After last 75 Days and before last 15 days View

In [17]:
df['after_75_before_last_15_days_users_view_value'] = df['cbrt_after_75_before_last_15_days_users_view']/ df['avg_after_75_before_last_15_days_users_view']
df['after_75_before_last_15_days_users_view_value'].replace([np.inf, -np.inf], 0, inplace=True)
df['after_75_before_last_15_days_users_view_value'] = df['after_75_before_last_15_days_users_view_value']/ df['after_75_before_last_15_days_users_view_value'].max()
df['after_75_before_last_15_days_users_view_weighted'] = df['after_75_before_last_15_days_users_view_value'] * after_75_before_last_15_days_users_view_weighted


/var/folders/7w/pnb04fl57d30_b6_35c_x4xwzktnc0/T/ipykernel_7346/3922326336.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['after_75_before_last_15_days_users_view_value'].replace([np.inf, -np.inf], 0, inplace=True)


### Variable 7 - Last 15 Days Sale

In [18]:
df['last_15_days_sale_value'] = df['cbrt_last_15_days_sale']/ df['avg_last_15_days_sale']
df['last_15_days_sale_value'].replace([np.inf, -np.inf], 0, inplace=True)
df['last_15_days_sale_value'] = df['last_15_days_sale_value']/ df['last_15_days_sale_value'].max()
df['last_15_days_sale_weighted'] = df['last_15_days_sale_value']*last_15_days_sale_weighted


/var/folders/7w/pnb04fl57d30_b6_35c_x4xwzktnc0/T/ipykernel_7346/1883313845.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['last_15_days_sale_value'].replace([np.inf, -np.inf], 0, inplace=True)


### Variable 8 - After last 75 days and before 15 days Sale

In [19]:
df['after_75_before_last_15_days_sale_value'] = df['cbrt_after_75_before_last_15_days_sale']/ df['avg_after_75_before_last_15_days_sale']
df['after_75_before_last_15_days_sale_value'].replace([np.inf, -np.inf], 0, inplace=True)

df['after_75_before_last_15_days_sale_value'] = df['after_75_before_last_15_days_sale_value']/ df['after_75_before_last_15_days_sale_value'].max()
df['after_75_before_last_15_days_sale_weighted'] = df['after_75_before_last_15_days_sale_value'] * after_75_before_last_15_days_sale_weighted

/var/folders/7w/pnb04fl57d30_b6_35c_x4xwzktnc0/T/ipykernel_7346/1927668508.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['after_75_before_last_15_days_sale_value'].replace([np.inf, -np.inf], 0, inplace=True)


### Variable 9 :- RPV (Revenue Per View ) - Last 15 Days Revenue Only

In [20]:
df['RPV_value'] = df['cbrt_last_15_days_revenue'] / df['cbrt_last_15_days_users_view']
df['RPV_value'].replace([np.inf, -np.inf], 0, inplace=True)
df['RPV_value'] = df['RPV_value'] / df['RPV_value'].max()
df['RPV_weighted'] = df['RPV_value'] *  RPV_weighted


/var/folders/7w/pnb04fl57d30_b6_35c_x4xwzktnc0/T/ipykernel_7346/1619000769.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['RPV_value'].replace([np.inf, -np.inf], 0, inplace=True)


### Variable 10 :- SPV ( Sales Per View) - Last 15 Days Sales Only

In [21]:
df['SPV_value'] = df['cbrt_last_15_days_sale']/ df['cbrt_last_15_days_users_view']
df['SPV_value'].replace([np.inf, -np.inf], 0, inplace=True)
df['SPV_value'] = df['SPV_value'] / df['SPV_value'].max()
df['SPV_weighted'] = df['SPV_value'] * SPV_weighted

/var/folders/7w/pnb04fl57d30_b6_35c_x4xwzktnc0/T/ipykernel_7346/2290732034.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['SPV_value'].replace([np.inf, -np.inf], 0, inplace=True)


# <h2> Calculating Rank </h2>

In [22]:
df['inventory_age_rank'] = df['product_aging_weighted'] + df['inventory_weighted'] + df['last_15_days_revenue_weighted_1']

df['inventory_age_rank'] = np.where(df['product_aging'] > 20, 0 , df['inventory_age_rank'])


df['rest_variable_rank'] = df['last_15_days_revenue_weighted_2'] + df['after_75_before_last_15_days_revenue_weighted'] + df['last_15_days_users_view_weighted'] + df['after_75_before_last_15_days_users_view_weighted'] + df['last_15_days_sale_weighted'] + df['after_75_before_last_15_days_sale_weighted'] + df['RPV_weighted'] + df['SPV_weighted']
df['Rank'] =  df[['inventory_age_rank', 'rest_variable_rank']].max(axis=1)



In [23]:

df['Rank_v2'] = np.where((df['item_price'] <= 500) & (df['item_code'].str.startswith('COMBO_') == True)  , (df['Rank'] - (0.40*df['Rank'])) , df['Rank'])

In [24]:
# # Testing of item ids
# check = np.where(df['item_code'] == "1128481")
# check

In [25]:
Nail_Out_way = ["1128481","1128486","1127519","1128484","1127500","1127525","1127508","1127467",
                "1127468","1127569","1127523","1127438","1127491","1127516","1127497","1127554",
                "1127532","1127527","1127570","1127511","1127536","1127474","1127504","1127520",
                "1127502","1127408","1127431","1127436","1127490","1127567","1127394","1127395",
                "1127396","1127397","1127398","1127399","1127400","1127401","1127402","1127403",
                "1127404","1127405","1127406","1127407","1127409","1127410","1127411","1127412",
                "1127413","1127414","1127415","1127416","1127417","1127418","1127419","1127420",
                "1127421","1127422","1127423","1127424","1127425","1127426","1127428","1127429",
                "1127430","1127432","1127434","1127435","1127437","1127439","1127454","1127455",
                "1127456","1127457","1127458","1127459","1127460","1127461","1127462","1127463",
                "1127464","1127465","1127466","1127469","1127470","1127471","1127472","1127473",
                "1127475","1127476","1127477","1127478","1127479","1127480","1127481","1127482",
                "1127483","1127484","1127485","1127486","1127487","1127488","1127489","1127492",
                "1127493","1127494","1127495","1127496","1127498","1127499","1127501","1127503",
                "1127505","1127506","1127507","1127509","1127510","1127512","1127513","1127514",
                "1127515","1127517","1127518","1127521","1127522","1127524","1127526","1127528",
                "1127529","1127530","1127531","1127533","1127534","1127535","1127537","1127538",
                "1127539","1127540","1127541","1127542","1127543","1127544","1127545","1127546",
                "1127547","1127548","1127549","1127550","1127551","1127552","1127553","1127555",
                "1127556","1127557","1127558","1127559","1127560","1127561","1127562","1127563",
                "1127564","1127565","1127566","1127578","1127579","1127580","1127581","1127582",
                "1127583","1127584","1127585","1127586","1127587","1127588","1127589","1127590",
                "1128482","1128483","1128485","1131299"]


FE_list = ["1105599","1123775","1102694","1102690","1105600","1105656",
           "1102697","1102691","1102695","1102715","1102702","1102704",
           "1102734","1105598","1132374","1132375","1132376",
           "1132395","1132396","1132377","1132378","1132397","1132398",
           "1132379","1132380","1132381","1132382","1132383","1132399",
           "1132384","1132400","1132385","1132386",
           "1132387","1132388","1132389","1132390","1132391","1132392",
           "1132393","1132394","1132401","1132402","1132403","1132404"]

final_list = Nail_Out_way + FE_list


In [26]:
df_subset = df.copy()
df_subset_2 = df.copy()

# Reducing the ranks of below item ids by 40%

### Subsetting Second Data Frame
df_def = df_subset.loc[df_subset['item_code'].isin(FE_list)]
df_def['rank_v3'] = df_def['Rank_v2']- (0.40*df_def['Rank_v2'])
print(df_def.shape)

df_nail_our_way = df_subset_2.loc[df_subset_2['item_code'].isin(Nail_Out_way)]
df_nail_our_way['rank_v3'] = df_nail_our_way['Rank_v2']+ (0.60*df_nail_our_way['Rank_v2'])
print(df_nail_our_way.shape)



### Subsetting First Data Frame
df = df.loc[~df['item_code'].isin(final_list)]
df['rank_v3'] = df['Rank_v2'] 
print(df.shape)



### Union Both Data Frame
df = pd.concat([df, df_def,df_nail_our_way])

(18, 63)
(177, 63)
(55309, 63)


/var/folders/7w/pnb04fl57d30_b6_35c_x4xwzktnc0/T/ipykernel_7346/719711171.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_def['rank_v3'] = df_def['Rank_v2']- (0.40*df_def['Rank_v2'])
/var/folders/7w/pnb04fl57d30_b6_35c_x4xwzktnc0/T/ipykernel_7346/719711171.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_nail_our_way['rank_v3'] = df_nail_our_way['Rank_v2']+ (0.60*df_nail_our_way['Rank_v2'])
/var/folders/7w/pnb04fl57d30_b6_35c_x4xwzktnc0/T/ipykernel_7346/719711171.py:19: SettingWithCopyWarning:

In [27]:
# df.to_csv(folder_path_out_v0)

In [28]:
df_final = df[['item_id','item_code','item_name','brandname','item_price','categoryl1','categoryl2','categoryl3','product_aging','inventory_left','last_15_days_users_view','after_75_before_last_15_days_users_view','last_15_days_sale','after_75_before_last_15_days_sale','last_15_days_revenue','after_75_before_last_15_days_revenue','SPV_value',
                'RPV_value','inventory_age_rank','rest_variable_rank','Rank','Rank_v2','rank_v3','Social Nudge']]


df_final['New_rank'] = df_final['rank_v3'].rank(ascending=False, method='first')


/var/folders/7w/pnb04fl57d30_b6_35c_x4xwzktnc0/T/ipykernel_7346/2083027865.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['New_rank'] = df_final['rank_v3'].rank(ascending=False, method='first')


In [29]:
# df_final[df_final['item_id']== 7578060].head(10)

In [30]:
df_RelevancyLogicV1 = df_final[['item_id', 'item_code','brandname','categoryl1','New_rank','Social Nudge']]

df_RelevancyLogicV1.rename(columns={'item_code': 'Item Code', 'brandname': 'Brand', 'categoryl1' : 'L1','New_rank':'Product Priority'}, inplace=True)




/var/folders/7w/pnb04fl57d30_b6_35c_x4xwzktnc0/T/ipykernel_7346/998635896.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_RelevancyLogicV1.rename(columns={'item_code': 'Item Code', 'brandname': 'Brand', 'categoryl1' : 'L1','New_rank':'Product Priority'}, inplace=True)


<h1><center> Changing In the Relevancy File </center> </h1>

## 1. Brand Boosting
## Importing Data From Google Sheets

In [31]:
### This code will import brands from the user
def convert_google_sheet_url(url):
    # Regular expression to match and capture the necessary part of the URL
    pattern = r'https://docs\.google\.com/spreadsheets/d/([a-zA-Z0-9-_]+)(/edit#gid=(\d+)|/edit.*)?'

    # Replace function to construct the new URL for CSV export
    # If gid is present in the URL, it includes it in the export URL, otherwise, it's omitted
    replacement = lambda m: f'https://docs.google.com/spreadsheets/d/{m.group(1)}/export?' + (f'gid={m.group(3)}&' if m.group(3) else '') + 'format=csv'

    # Replace using regex
    new_url = re.sub(pattern, replacement, url)

    return new_url

# Google Sheet Url 
# url = 'https://docs.google.com/spreadsheets/d/16Tj2ScTIBTLF4vfUfiZ03j_uY-AZmPfXJCgq_1VmNdA/edit?gid=0'
url = 'https://docs.google.com/spreadsheets/d/16Tj2ScTIBTLF4vfUfiZ03j_uY-AZmPfXJCgq_1VmNdA/edit?gid=0'
new_url = convert_google_sheet_url(url)

print("Google Sheet URL :- ",new_url)


response = requests.get(new_url, verify=False)
response.raise_for_status() 

df_gsheets = pd.read_csv(io.StringIO(response.text), index_col=False)

df_brand_boost = df_gsheets['BrandBoost'].tolist()
print("Brands to Boost:", df_brand_boost)

# df_gsheets = pd.read_csv(new_url,index_col=False)


# ### Converting User input dataframe to list
# df_brand_boost = df_gsheets['BrandBoost'].tolist()

# print("Brands to Boost" ,df_brand_boost )

Google Sheet URL :-  https://docs.google.com/spreadsheets/d/16Tj2ScTIBTLF4vfUfiZ03j_uY-AZmPfXJCgq_1VmNdA/export?format=csv


/Users/shubham.bansla/anaconda3/lib/python3.10/site-packages/urllib3/connectionpool.py:1045: InsecureRequestWarning: Unverified HTTPS request is being made to host 'docs.google.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/Users/shubham.bansla/anaconda3/lib/python3.10/site-packages/urllib3/connectionpool.py:1045: InsecureRequestWarning: Unverified HTTPS request is being made to host 'doc-04-9s-sheets.googleusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


Brands to Boost: ['Akind']


In [32]:
## Reading Product Ranking File 
# folder_path_in = folder_path_out
# df_RelevancyLogic = pd.read_csv(folder_path_in, index_col=False)
df_RelevancyLogic = df_RelevancyLogicV1
df_RelevancyLogic = df_RelevancyLogic.drop(df_RelevancyLogic.columns[0], axis=1)
df_RelevancyLogic.head(2)

,Item Code,Brand,L1,Product Priority,Social Nudge
0,COMBO_260509121915147-IO65,Aqualogica,Skin,2989.0,
1,COMBO_260509121915148-L72G,Aqualogica,Skin,2993.0,


In [33]:
df = df_RelevancyLogic

# DataFrame containing brands that should have their ranks increased
priority_brands = df_brand_boost

# Function to boost the rank of selected brands
def boost_rank(df, priority_brands, boost_percentage=0.60):
    df.loc[df['Brand'].isin(priority_brands), 'Product Priority'] *= (1 - boost_percentage)
    df['Product Priority'] = df['Product Priority'].round()
    #.astype(int)  # Ensure integer values
    df = df.sort_values(by='Product Priority', ascending=True).reset_index(drop=True)  # Re-sort after updates
    
    # Renumber the Product Priority from 1 again
    df['Product Priority'] = range(1, len(df) + 1)
    return df


# Apply rank boosting
df_BrandBoosted = boost_rank(df, priority_brands)

In [34]:
### Removing all the Items Having Category L1
df_BrandBoosted = df_BrandBoosted[df_BrandBoosted["L1"] != "Free-L1"].reset_index(drop=True)

<h1><center> Removing Clustering of Same Brands </center></h1>

### Method 1:- Simpy increase the rank by 2 

In [35]:
### Removing Clutering of Brands 

for i in range(1, len(df_BrandBoosted)-1):
    if df_BrandBoosted.loc[i, "Brand"] == df_BrandBoosted.loc[i + 1, "Brand"]:
        df_BrandBoosted.loc[i+1, "Product Priority"] += 2
        
df_BrandBoosted["Product Priority"] = df_BrandBoosted["Product Priority"].rank(method="first")
# .astype(int)

df_BrandBoosted = df_BrandBoosted.sort_values(by='Product Priority', ascending=True).reset_index(drop=True)


In [36]:
df_Ranking = df_BrandBoosted

<h1><center> Storing Final Prepared Ranking File In Local System </center></h1>

In [37]:

### For testing

print("Brands Boosted",df_brand_boost)

# df_final.to_csv(folder_path_out_v1)
# print("Raw File Store to Local System")


# df_RelevancyLogicV1.to_csv(folder_path_out_V2)
# print("Ranking File Before Clustering")


df_Ranking.to_csv(folder_path_out_prepared)
print("Final Prepared File Store to local System")

df_seo = df_final[['item_id', 'item_code','brandname','categoryl1','categoryl2','categoryl3','New_rank']]
df_seo.to_csv(folder_path_out_seo)
print("SEO File Store to local System")


df_nudge = df_Ranking[
    df_Ranking['Social Nudge'].notna() &
    (df_Ranking['Social Nudge'].astype(str).str.strip() != '')
]
df_nudge = df_nudge[['Item Code','Brand','L1','Social Nudge']]
df_nudge.to_csv(folder_path_out_nudge)
print("Social Nudge File Is Saved")


Brands Boosted ['Akind']
Final Prepared File Store to local System
SEO File Store to local System
Social Nudge File Is Saved


In [38]:
print('Script Finish')

Script Finish
